# 🌌 Electromagnetic Plane Wave Simulation

This notebook builds an **interactive 3D visualisation** of an electromagnetic plane wave using VPython.

You will:

- Construct the simulation step by step  
- Understand how E and B fields behave  
- Observe energy flow via the Poynting vector  
- Interact with the system using sliders

### ⚠ NOTE: "Next step" will only execute if you run the "previous step".
> Because the imports, variable declaration, etc., are written in separate cells to teach you the steps.

---

## step 0: Importing

In [1]:
#imports
from vpython import *
import numpy as np

C:\Users\ABDUL JABBAR KHAN\AppData\Local\Programs\Python\Python314\Lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

---

## 🧱 Step 1: Setting Up the Visual Space

Before we visualize electromagnetic waves, we need a **3D environment**.

In this step, we will:

- Create a **canvas** (our simulation space)
- Add **coordinate axes** (X, Y, Z)
- Label directions for clarity

🧠 *Think of this as building the coordinate system where physics will come alive.*

In [2]:
#NOTE: Run this cell after running the above cell (i.e, imports)

#Setup the Canvas
scene_color = vec(0.788, 0.807, 0.925)
scene = canvas(title='Electromagnetic Plane Wave\n', width=1000, height=450, background=scene_color, pos=vec(0,1,1))

axis_length = 12

#Axes
arrow(pos=vec(0,0,0), axis=vec(axis_length,0,0), color=color.black, shaftwidth=0.05) 
arrow(pos=vec(0,0,0), axis=vec(0,axis_length,0), color=color.black, shaftwidth=0.05)
arrow(pos=vec(0,0,0), axis=vec(0,0,axis_length), color=color.black, shaftwidth=0.05)

#Axis Labels
label(pos=vec(axis_length + 0.5, 0, 0), text='X', box=False, opacity=0, color=color.black)
label(pos=vec(0, axis_length + 0.5, 0), text='Y', box=False, opacity=0, color=color.black)
label(pos=vec(0, 0, axis_length + 0.5), text='Z', box=False, opacity=0, color=color.black)

#Color Legend
label(pos=vec(0, 11.5, 0), text='Blue: Electric Field (E)', color=vec(0.103, 0.464, 1.000), box=True)
label(pos=vec(0, 10, 0), text='Red: Magnetic Field (B)', color=vec(1.000, 0.046, 0.380), box=True)
label(pos=vec(0, 8.5, 0), text='Yellow: Poynting Vector (S)', color=vec(1.000, 0.555, 0.000), box=True)

# Initial Wave Parameters
A = 2.0  
C = 2.0 
lam = 5.0 # Wavelength
k = 2 * np.pi / lam 
w = 2.0

<IPython.core.display.Javascript object>

---

## 🎛️ Step 2: Adding Interactive Controls

A simulation becomes powerful when you can **control it**.

Here, we introduce sliders to manipulate:

- Amplitude → strength of the wave  
- Wavelength → spatial variation  
- Frequency → speed of oscillation  


⚠️ For now, we use a temporary canvas just to test the sliders.

In [3]:
canvas()#just added an empty canvas to show the widgets below.

#Functions for the Interactive Sliders
scene.append_to_caption('\n--- Wave Controls ---\n\n')

def update_amplitude(s):
    global A, C
    A = s.value
    C = s.value #NOTE:  Keep B-field scaled equally for the visual
    amp_text.text = f' Amplitude: {A:.1f}\n\n'

def update_wavelength(s):
    global k, lam
    lam = s.value
    k = 2 * np.pi / lam
    wl_text.text = f' Wavelength: {lam:.1f}\n\n'

def update_frequency(s):
    global w
    w = s.value
    freq_text.text = f' Frequency (w): {w:.1f}\n\n'

#Create the sliders and their text labels
scene.append_to_caption('Amplitude: ')
amp_slider = slider(min=0.5, max=4.0, value=2.0, bind=update_amplitude, length=300)
amp_text = wtext(text=f' Amplitude: {A:.1f}\n\n')

scene.append_to_caption('Wavelength: ')
wl_slider = slider(min=2.0, max=10.0, value=5.0, bind=update_wavelength, length=300)
wl_text = wtext(text=f' Wavelength: {lam:.1f}\n\n')

scene.append_to_caption('Frequency: ')
freq_slider = slider(min=0.5, max=5.0, value=2.0, bind=update_frequency, length=300)
freq_text = wtext(text=f' Frequency (w): {w:.1f}\n\n')

<IPython.core.display.Javascript object>

---

## ⚡Step 3: Representing Fields in Space

Now we introduce the core components of an electromagnetic wave:
```
- 🔵 Electric Field (E)  
- 🔴 Magnetic Field (B)  
- 🟡 Poynting Vector (S) — energy flow (Yellow Arrows **along the Z-axis**)
```

   *At this stage, they are `static` — we will `animate` them in the next step.*

In [4]:
canvas()#just added an empty canvas to show the arrows.

#Create the arrows with default magnitudes; we will update them later
E_arrows = []
B_arrows = []
S_arrows = [] 

for z in np.arange(0, axis_length, 0.5):
    e_arr = arrow(pos=vec(0,0,z), axis=vec(0,A,0), color=vec(0.103, 0.464, 1.000), shaftwidth=0.05)
    E_arrows.append(e_arr)
    b_arr = arrow(pos=vec(0,0,z), axis=vec(-C,0,0), color=vec(1.000, 0.046, 0.380), shaftwidth=0.05)
    B_arrows.append(b_arr)
    s_arr = arrow(pos=vec(0,0,z), axis=vec(0,0,1), color=vec(1.000, 0.555, 0.000), shaftwidth=0.08)
    S_arrows.append(s_arr)

#Modify the magnitudes of the arrows as a function of time (according to the equations).
t = 0
dt = 0.05

<IPython.core.display.Javascript object>

---
### Step: 4 Equations😅

👉 We **animate the fields over time** , by varying their magnitudes according to the equations (assuming wave travelling in Z-axis).

$$\vec{E}(z,t) = \hat{j} E_0 \sin(kz - \omega t)$$
$$\vec{B}(z,t) = -\hat{i} B_0 \sin(kz - \omega t)$$

This is where the physics truly emerges:
```
- Fields start oscillating  
- Energy begins to flow  (see: Poynting vector)
```

In [ ]:
canvas()#just added an empty canvas to show the arrows.

#Create the arrows with default magnitudes, we will update them later
E_arrows = []
B_arrows = []
S_arrows = [] 

for z in np.arange(0, axis_length, 0.5):
    e_arr = arrow(pos=vec(0,0,z), axis=vec(0,A,0), color=vec(0.103, 0.464, 1.000), shaftwidth=0.05)
    E_arrows.append(e_arr)
    b_arr = arrow(pos=vec(0,0,z), axis=vec(-C,0,0), color=vec(1.000, 0.046, 0.380), shaftwidth=0.05)
    B_arrows.append(b_arr)
    s_arr = arrow(pos=vec(0,0,z), axis=vec(0,0,1), color=vec(1.000, 0.555, 0.000), shaftwidth=0.08)
    S_arrows.append(s_arr)

#Modify the magnitudes of the arrows as a function of time (according to the equations).
t = 0
dt = 0.05

#NOTE: infinite while loop is replaced by for loop for running purposes
for _ in range(10000):  # runs long but not infinite
    rate(30)
    
    for i in range(len(E_arrows)):
        z = E_arrows[i].pos.z 
        
        E_y = A * np.sin(k * z - w * t)
        B_x = -C * np.sin(k * z - w * t)
        
        E_vec = vec(0, E_y, 0)
        B_vec = vec(B_x, 0, 0)
        
        E_arrows[i].axis = E_vec
        B_arrows[i].axis = B_vec
        S_arrows[i].axis = cross(E_vec, B_vec) * 0.3
        
    t += dt

<IPython.core.display.Javascript object>

---

## 🚀 Step 5: Full Simulation

Now we combine everything👇:


In [ ]:
from vpython import *
import numpy as np

#Setup the Canvas
scene_color = vec(0.788, 0.807, 0.925)
scene = canvas(title='Electromagnetic Plane Wave\n', width=1000, height=450, background=scene_color, pos=vec(0,1,1))

axis_length = 12

#Axes
arrow(pos=vec(0,0,0), axis=vec(axis_length,0,0), color=color.black, shaftwidth=0.05) 
arrow(pos=vec(0,0,0), axis=vec(0,axis_length,0), color=color.black, shaftwidth=0.05)
arrow(pos=vec(0,0,0), axis=vec(0,0,axis_length), color=color.black, shaftwidth=0.05)

#Axis Labels
label(pos=vec(axis_length + 0.5, 0, 0), text='X', box=False, opacity=0, color=color.black)
label(pos=vec(0, axis_length + 0.5, 0), text='Y', box=False, opacity=0, color=color.black)
label(pos=vec(0, 0, axis_length + 0.5), text='Z', box=False, opacity=0, color=color.black)

#Color Legend
label(pos=vec(0, 11.5, 0), text='Blue: Electric Field (E)', color=vec(0.103, 0.464, 1.000), box=True)
label(pos=vec(0, 10, 0), text='Red: Magnetic Field (B)', color=vec(1.000, 0.046, 0.380), box=True)
label(pos=vec(0, 8.5, 0), text='Yellow: Poynting Vector (S)', color=vec(1.000, 0.555, 0.000), box=True)

# Initial Wave Parameters
A = 2.0  
C = 2.0 
lam = 5.0 # Wavelength
k = 2 * np.pi / lam 
w = 2.0  

#Functions for Interactive Sliders
scene.append_to_caption('\n--- Wave Controls ---\n\n')

def update_amplitude(s):
    global A, C
    A = s.value
    C = s.value #NOTE:  Keep B-field scaled equally for the visual
    amp_text.text = f' Amplitude: {A:.1f}\n\n'

def update_wavelength(s):
    global k, lam
    lam = s.value
    k = 2 * np.pi / lam
    wl_text.text = f' Wavelength: {lam:.1f}\n\n'

def update_frequency(s):
    global w
    w = s.value
    freq_text.text = f' Frequency (w): {w:.1f}\n\n'

#Create the sliders and their text labels
scene.append_to_caption('Amplitude: ')
amp_slider = slider(min=0.5, max=4.0, value=2.0, bind=update_amplitude, length=300)
amp_text = wtext(text=f' Amplitude: {A:.1f}\n\n')

scene.append_to_caption('Wavelength: ')
wl_slider = slider(min=2.0, max=10.0, value=5.0, bind=update_wavelength, length=300)
wl_text = wtext(text=f' Wavelength: {lam:.1f}\n\n')

scene.append_to_caption('Frequency: ')
freq_slider = slider(min=0.5, max=5.0, value=2.0, bind=update_frequency, length=300)
freq_text = wtext(text=f' Frequency (w): {w:.1f}\n\n')

#Create the arrows with default magnitudes, we will update them later
E_arrows = []
B_arrows = []
S_arrows = [] 

for z in np.arange(0, axis_length, 0.5):
    e_arr = arrow(pos=vec(0,0,z), axis=vec(0,A,0), color=vec(0.103, 0.464, 1.000), shaftwidth=0.05)
    E_arrows.append(e_arr)
    b_arr = arrow(pos=vec(0,0,z), axis=vec(-C,0,0), color=vec(1.000, 0.046, 0.380), shaftwidth=0.05)
    B_arrows.append(b_arr)
    s_arr = arrow(pos=vec(0,0,z), axis=vec(0,0,1), color=vec(1.000, 0.555, 0.000), shaftwidth=0.08)
    S_arrows.append(s_arr)

#Modify the magnitudes of the arrows as a function of time (according to the equations).
t = 0
dt = 0.05

#NOTE: infinite "while loop" is replaced by a "for loop" for running purposes in binder
for _ in range(10000):  # runs long but not infinite
    rate(30)
    
    for i in range(len(E_arrows)):
        z = E_arrows[i].pos.z 
        
        E_y = A * np.sin(k * z - w * t)
        B_x = -C * np.sin(k * z - w * t)
        
        E_vec = vec(0, E_y, 0)
        B_vec = vec(B_x, 0, 0)
        
        E_arrows[i].axis = E_vec
        B_arrows[i].axis = B_vec
        S_arrows[i].axis = cross(E_vec, B_vec) * 0.3
        
    t += dt


C:\Users\ABDUL JABBAR KHAN\AppData\Local\Programs\Python\Python314\Lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## 🧠 What You Should Notice
As the simulation runs:

- Fields remain **perpendicular**  
- They oscillate **in phase**  
- Energy flow **pulsates**, not constant

```
 See the README.md file of this simulation to know more.
```
  ---

#  HAPPY LEARNING :)